# Retrieval Augmented Generation (RAG) with Langchain
*Using IBM Granite Models*

## In this notebook
This notebook contains instructions for performing Retrieval Augumented Generation (RAG). RAG is an architectural pattern that can be used to augment the performance of language models by recalling factual information from a knowledge base, and adding that information to the model query. The most common approach in RAG is to create dense vector representations of the knowledge base in order to retrieve text chunks that are semantically similar to a given user query.

RAG use cases include:
- Customer service: Answering questions about a product or service using facts from the product documentation.
- Domain knowledge: Exploring a specialized domain (e.g., finance) using facts from papers or articles in the knowledge base.
- News chat: Chatting about current events by calling up relevant recent news articles.

In its simplest form, RAG requires 3 steps:

- Initial setup:
  - Index knowledge-base passages for efficient retrieval. In this recipe, we take embeddings of the passages and store them in a vector database.
- Upon each user query:
  - Retrieve relevant passages from the database. In this recipe, we use an embedding of the query to retrieve semantically similar passages.
  - Generate a response by feeding retrieved passage into a large language model, along with the user query.

## Setting up the environment

### Python version

Ensure you are running Python 3.10 or 3.11.

In [2]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 12), "Use Python 3.10 or 3.11 to run this notebook."

### Install dependencies

Granite Kitchen comes with a bundle of dependencies that are required for notebooks. See the list of packages in its [`setup.py`](https://github.com/ibm-granite-community/granite-kitchen/blob/main/setup.py). 

In [3]:
! pip install "git+https://github.com/alanbraz/granite-kitchen.git" "langchain-huggingface" "langchain-milvus" "wget"

  Cloning https://github.com/alanbraz/granite-kitchen.git to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-22axcrd1
  Running command git clone --filter=blob:none --quiet https://github.com/alanbraz/granite-kitchen.git /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-req-build-22axcrd1
  Resolved https://github.com/alanbraz/granite-kitchen.git to commit b5f7e403f3fffa7f6391a005655e70670c415f1f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/ibm-granite-community/utils to /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-install-hwyiknfi/ibm-granite-community-utils_1d3e61c9753642fca4e5a86af96d8a5a
  Running command git clone --filter=blob:none --quiet https://github.com/ibm-granite-community/utils /private/var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/pip-install-hwyiknfi/ibm-granite-community-utils_1d3e61c9753642fca4e5a86a

### Serving the Granite AI model


This notebook requires IBM Granite models to be served by an AI model runtime so that the models can be invoked or called. This notebook can use a locally accessible [Ollama](https://github.com/ollama/ollama) server to serve the models, or the [Replicate](https://replicate.com) cloud service.

During the pre-work, you may have either started a local Ollama server on your computer, or setup Replicate access and obtained an [API token](https://replicate.com/account/api-tokens).

## Selecting System Components

### Choose your Embeddings Model

Specify the model to use for generating embedding vectors from text.

To use a model from a provider other than Huggingface, replace this code cell with one from [this Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

In [4]:
from langchain_ollama import OllamaEmbeddings
embeddings_model = OllamaEmbeddings(model="snowflake-arctic-embed2")

### Choose your Vector Database

Specify the database to use for storing and retrieving embedding vectors.

To connect to a vector database other than Milvus substitute this code cell with one from [this Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [5]:
from langchain_milvus import Milvus
import tempfile

db_file = tempfile.NamedTemporaryFile(prefix="milvus_br_ollama", suffix=".db", delete=False).name
print(f"The vector database will be saved to {db_file}")

vector_db = Milvus(embedding_function=embeddings_model, connection_args={"uri": db_file}, auto_id=True)

The vector database will be saved to /var/folders/rq/kq3jfvfd3_q5zcg4bfx907fh0000gn/T/milvus_br_ollamaipjllwlj.db


## Select your model


Select a Granite model to use. Here we use a Langchain client to connect to the model. If there is a locally accessible Ollama server, we use an Ollama client to access the model. Otherwise, we use a Replicate client to access the model.

When using Replicate, if the `REPLICATE_API_TOKEN` environment variable is not set, or a `REPLICATE_API_TOKEN` Colab secret is not set, then the notebook will ask for your [Replicate API token](https://replicate.com/account/api-tokens) in a dialog box.

In [6]:
import os
import requests
from langchain_ollama.llms import OllamaLLM
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import set_env_var

try: # Look for a locally accessible Ollama server for the model
    response = requests.get(os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434"))
    model = OllamaLLM(model="granite3-dense:8b")
except Exception: # Use Replicate for the model
    set_env_var("REPLICATE_API_TOKEN")
    model = Replicate(model="ibm-granite/granite-3.0-8b-instruct")


## Building the Vector Database

In this example, we take the State of the Union speech text, split it into chunks, derive embedding vectors using the embedding model, and load it into the vector database for querying.

### Download the document

Here we use President Biden's State of the Union address from March 1, 2022.

In [8]:
# import wget

# filename = 'silvio_santos.pdf'
# url = 'https://pt.wikipedia.org/w/index.php?title=Especial:DownloadAsPdf&page=Silvio_Santos&action=show-download-screen'

# if not os.path.isfile(filename):
#   wget.download(url, out=filename)

### Split the document into chunks

Split the document into text segments that can fit into the model's context window.

In [7]:
! pip install unstructured

from langchain.document_loaders import UnstructuredURLLoader
from langchain.text_splitter import CharacterTextSplitter

loader = UnstructuredURLLoader([ "https://pt.wikipedia.org/wiki/Silvio_Santos" ])
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

Created a chunk of size 804, which is longer than the specified 300
Created a chunk of size 707, which is longer than the specified 300
Created a chunk of size 1173, which is longer than the specified 300
Created a chunk of size 720, which is longer than the specified 300
Created a chunk of size 770, which is longer than the specified 300
Created a chunk of size 774, which is longer than the specified 300
Created a chunk of size 301, which is longer than the specified 300
Created a chunk of size 649, which is longer than the specified 300
Created a chunk of size 746, which is longer than the specified 300
Created a chunk of size 370, which is longer than the specified 300
Created a chunk of size 492, which is longer than the specified 300
Created a chunk of size 786, which is longer than the specified 300
Created a chunk of size 388, which is longer than the specified 300
Created a chunk of size 305, which is longer than the specified 300
Created a chunk of size 597, which is longer th

### Populate the vector database

NOTE: Population of the vector database may take over a minute depending on your embedding model and service.

In [8]:
vector_db.add_documents(texts)

[454431814359973888,
 454431814359973889,
 454431814359973890,
 454431814359973891,
 454431814359973892,
 454431814359973893,
 454431814359973894,
 454431814359973895,
 454431814359973896,
 454431814359973897,
 454431814359973898,
 454431814359973899,
 454431814359973900,
 454431814359973901,
 454431814359973902,
 454431814359973903,
 454431814359973904,
 454431814359973905,
 454431814359973906,
 454431814359973907,
 454431814359973908,
 454431814359973909,
 454431814359973910,
 454431814359973911,
 454431814359973912,
 454431814359973913,
 454431814359973914,
 454431814359973915,
 454431814359973916,
 454431814359973917,
 454431814359973918,
 454431814359973919,
 454431814359973920,
 454431814359973921,
 454431814359973922,
 454431814359973923,
 454431814359973924,
 454431814359973925,
 454431814359973926,
 454431814359973927,
 454431814359973928,
 454431814359973929,
 454431814359973930,
 454431814359973931,
 454431814359973932,
 454431814359973933,
 454431814359973934,
 454431814359

## Querying the Vector Database

### Conduct a similarity search

Search the database for similar documents by proximity of the embedded vector in vector space.

In [9]:
query = "Quando Silvio Santos morreu?"
docs = vector_db.similarity_search(query)
print(docs[0].page_content)

Morte

Ver artigo principal: Morte de Silvio Santos


## Answering Questions

### Automate the RAG pipeline

Build a RAG chain with the model and the document retriever. See the [Granite Prompting Guide](https://www.ibm.com/granite/docs/models/granite/#prompt-anatomy) for information on the prompt template.

In [10]:
from langchain.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Create a prompt template for question-answering with the retrieved context.
prompt_template = """\
<|start_of_role|>system<|end_of_role|>Você é um assistente brasileiro. Responda sempre em Português.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>Use os seguintes pedaços de contexto para responder à pergunta no final.
Se você não sabe a resposta, apenas diga que não sabe, não tente inventar uma resposta. 

{context}

Pergunta: {input}<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>"""

# Assemble the retrieval-augmented generation chain.
qa_chain_prompt = PromptTemplate.from_template(prompt_template)
combine_docs_chain = create_stuff_documents_chain(model, qa_chain_prompt)
rag_chain = create_retrieval_chain(vector_db.as_retriever(), combine_docs_chain)

### Generate a retrieval-augmented response to a question

Use the RAG chain to process the query. 

In [11]:
output = rag_chain.invoke({"input": "Quando Silvio Santos morreu?"})

print(output['answer'])

Silvio Santos faleceu no dia 17 de agosto de 2024, de acordo com os artigos publicados pela g1, SBT e O Globo. Ele morreu aos 93 anos em São Paulo.


In [12]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template_chat = ChatPromptTemplate([
    ("system", "Você é um assistente brasileiro. Responda sempre em Português."),
    ("user", "{input}")
])


# Assemble the retrieval-augmented generation chain.
model.invoke(prompt_template_chat.format(input="Quando Silvio Santos morreu?"))

'Desculpe, mas não tenho informações sobre o falecimento de Silvio Santos. Ele é um empresário e apresentador brasileiro, vivo e ativo na televisão. Se você tiver outras perguntas, por favor, me avise.'

In [19]:
for i in range(10):
    print("RAG:", rag_chain.invoke({"input": "Quando Silvio Santos nasceu?"})['answer']) # 12 de dezembro de 1930
    print("Puro:", model.invoke(prompt_template_chat.format(input="Quando Silvio Santos nasceu?")))

RAG: Silvio Santos nasceu em 20 de janeiro de 1930.
Puro: Silvio Santos nasceu em 13 de abril de 1930, em São Paulo, Brasil.
RAG: Silvio Santos nasceu em 20 de janeiro de 1930.
Puro: Silvio Santos nascida em 1930, no estado de São Paulo, Brasil.
RAG: Silvio Santos nasceu em 12 de janeiro de 1930.
Puro: Silvio Santos nasceu em 1 de janeiro de 1930, no município de São Paulo, Brasil.
RAG: Silvio Santos nasceu em 23 de novembro de 1930.
Puro: Silvio Santos nasceu em 23 de setembro de 1930, no município de São Paulo, no estado de São Paulo, Brasil.
RAG: Silvio Santos nasceu em 20 de setembro de 1930.
Puro: Silvio Santos nasceu em 18 de janeiro de 1930, no Rio de Janeiro, Brasil.
RAG: Silvio Santos nasceu em 20 de agosto de 1930.
Puro: Silvio Santos nasceram em 13 de agosto de 1930. Ele é um empresário, apresentador e proprietário do SBT, uma rede de televisão brasileira.
RAG: Silvio Santos nasceu em 2 de setembro de 1930.
Puro: Silvio Santos nascera em 16 de janeiro de 1930, em São Paulo, 

In [21]:
rag_chain.invoke({"input": "Quando nasceu Silvio Santos?"})

{'input': 'Quando nasceu Silvio Santos?',
 'context': [Document(metadata={'pk': 454431814359973900, 'source': 'https://pt.wikipedia.org/wiki/Silvio_Santos'}, page_content='Morte\n\nVer artigo principal: Morte de Silvio Santos'),
  Document(metadata={'pk': 454431814359974026, 'source': 'https://pt.wikipedia.org/wiki/Silvio_Santos'}, page_content='↑ «Prêmios - Silvio Santos». Sistema Brasileiro de Televisão. Consultado em 20 de agosto de 2024. Arquivado do original em 12 de novembro de 2014'),
  Document(metadata={'pk': 454431814359973972, 'source': 'https://pt.wikipedia.org/wiki/Silvio_Santos'}, page_content='Grupo Silvio Santos\n\nVer artigo principal: Grupo Silvio Santos'),
  Document(metadata={'pk': 454431814359973888, 'source': 'https://pt.wikipedia.org/wiki/Silvio_Santos'}, page_content='Silvio Santos\n\nالعربية\n\nŽemaitėška\n\nCatalà\n\nDeutsch\n\nEnglish\n\nEsperanto\n\nEspañol\n\nFrançais\n\nHausa\n\nעברית\n\nMagyar\n\nBahasa Indonesia\n\nIdo\n\nItaliano\n\n日本語\n\nქართული\n\nNe